In [1]:
import json
import os

with open('creds.json') as file:
  creds = json.load(file)

os.environ["HF_TOKEN"] = creds['HUGGINGFACEHUB_API_TOKEN']
os.environ["SERPAPI_API_KEY"] = creds["SERPAPI_API_KEY"]

In [3]:
! pip install -q langchain-huggingface

In [2]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [3]:
from langchain_core.prompts import PromptTemplate

In [4]:
from langchain_classic.agents import load_tools, Tool, AgentExecutor, AgentType, create_react_agent, initialize_agent
from langchain_classic import hub

In [17]:
from langchain_experimental.utilities import PythonREPL

/tmp/ipykernel_33294/1796928108.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.utilities import PythonREPL




### Завдання 1: Виклик LLM з базовим промптом

**Мета:** навчитися викликати LLM через LangChain зі звичайним текстовим промптом.

**Що потрібно зробити:**

1. Створіть промпт, який дозволяє отримати інформацію простою мовою на тему "Квантові обчислення". Відповідь моделі повинна містити визначення, ключові переваги та поточні дослідження в цій галузі.

2. Обмежте відповідь до 200 символів і пропишіть в промпті, аби відповідь була короткою (це зекономить вам час і гроші на згенеровані токени).

3. Встановіть своє значення температури на власний розсуд (тут немає правильного чи неправильного значення) і напишіть коментарем, чому ви обрали саме таке значення для цього завдання.

**Вибір моделі:** можна скористатись як моделлю з HuggingFace, так і ChatGPT будь-якої версії, яка вам до вподоби і пасує за прайсингом. В обох випадках потрібно імпортувати відповідний клас з LangChain для виклику LLM за API.

**Мова запитів:** промпти можна писати як українською, так і англійською — орієнтуйтесь на те, де і для чого ви хочете потім використовувати цей проєкт. У розв'язках промпти — українською.

---

**🔐 Як безпечно зберігати і підвантажувати API-ключі**

API-токен потрібно зчитувати з безпечного джерела, а **не хардкодити в ноутбуці**. Якщо хтось отримає доступ до вашого ключа, він буде витрачати токени за ваш рахунок, а вам це не треба :)

Є кілька способів. Перший ми використовували на лекції, ще два для розширення вашого розуміння, як ще це можна зробити і що шлях не лише один. Для виконання цього ДЗ можете використовувати будь-який спосіб підвантаження ключів у ноутбук.

**Спосіб 1: Файл `creds.json` (рекомендований)**

Створіть файл `creds.json` з вашими ключами, завантажте його в Google Colab під час роботи, але **не здавайте** цей файл у ДЗ і **не комітьте** в git.

```python
import json
with open("creds.json") as f:
    creds = json.load(f)
api_key = creds["HF_TOKEN"]
```

**Спосіб 2: Google Colab Secrets**

У лівій панелі Colab натисніть іконку 🔑 (Secrets) → "Add new secret" → введіть назву (наприклад, `HF_TOKEN`) та значення ключа → увімкніть тогл доступу для ноутбука.

```python
from google.colab import userdata
api_key = userdata.get("HF_TOKEN")
```

Зручно тим, що ключ зберігається в акаунті і доступний у всіх ваших ноутбуках. Мінус — при кожній новій сесії потрібно перевірити, що доступ увімкнено.

**Спосіб 3: Google AI Studio (для Gemini API)**

Якщо працюєте з моделями Google Gemini, отримати безкоштовний API-ключ можна в [Google AI Studio](https://aistudio.google.com/app/apikey): увійдіть з Google-акаунтом → натисніть "Get API key" → "Create API key". Далі використовуйте ключ через будь-який із способів вище.



In [8]:
llm = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-72B-Instruct",
    temperature = 0.1,
    max_new_tokens=200
)

chat = ChatHuggingFace(llm=llm)

In [6]:
text_prompt = "Briefly explain what quantum computing is, what its key advantages are, and what research is currently being conducted in this field. Your answer should be no more than 200 characters."

In [10]:
print(chat.invoke(text_prompt).content)

Quantum computing uses quantum bits (qubits) to process information beyond classical computing limits, offering exponential speedups for specific tasks. Research focuses on error correction, scalable qubit systems, and practical algorithm development.


Температура 0.1 вибрана для того, щоб модель дала якомога стислішу та впевнену відповідь та не витрачала ресурси для додаткової обробки та генерації відповіді. Загалом відповідь моделі хороша, лаконічна та вичерпна у контексті даного завдання.

### Завдання 2: Створення параметризованого промпта для генерації тексту
Тепер ми хочемо оновити попередній фукнціонал так, аби в промпт ми могли передавати тему як параметр. Для цього скористайтесь `PromptTemplate` з `langchain` і реалізуйте параметризований промпт та виклик моделі з ним.

Запустіть оновлений функціонал (промпт + модел) для пояснень про теми
- "Баєсівські методи в машинному навчанні"
- "Трансформери в машинному навчанні"
- "Explainable AI"

Виведіть результати відпрацювання моделі на екран.

In [12]:
llm_prompt = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-72B-Instruct",
    temperature = 0.1,
    max_new_tokens=200
)

chat_prompt = ChatHuggingFace(llm=llm_prompt)

prompt = PromptTemplate(
    template = "Briefly explain what {topic} is, what its key advantages are, and what research is currently being conducted in this field. Your answer should be no more than 200 characters.",
    input_variables = ["topic"]
)

In [13]:
topics = [
    "Bayesian Methods in Machine Learning",
    "Transformers in Machine Learning",
    "Explainable AI"
]

In [14]:
for topic in topics:
  text_prompt = prompt.format(topic = topic)
  print(topic)
  try:
    response = chat_prompt.invoke(text_prompt)
    print(response.content)
    print()
  except Exception as e:
    print(f"Error: {e}")

Bayesian Methods in Machine Learning
Bayesian Methods in ML use probability to model uncertainty in predictions. Advantages include handling small datasets and providing interpretable results. Current research focuses on scalable inference and deep Bayesian models.

Transformers in Machine Learning
Transformers are ML models excelling in handling sequential data, like text, through self-attention mechanisms. Advantages include parallelization and capturing long-range dependencies. Current research focuses on efficiency, scalability, and multimodal capabilities.

Explainable AI
Explainable AI (XAI) makes AI models' decisions understandable to humans, enhancing transparency and trust. Key advantages include improved accountability and usability. Current research focuses on developing more intuitive explanation methods and integrating XAI into various applications.



Можна побачити, що відповіді також лаконічні та вичерпні



### Завдання 3: Використання агента для автоматизації процесів
Створіть агента, який допоможе автоматично шукати інформацію про останні наукові публікації в різних галузях. Наприклад, агент має знайти 5 останніх публікацій на тему штучного інтелекту.

**Кроки:**
1. Налаштуйте агента типу ReAct в LangChain для виконання автоматичних запитів.
2. Створіть промпт, який спрямовує агента шукати інформацію в інтернеті або в базах даних наукових публікацій.
3. Агент повинен видати список публікацій, кожна з яких містить назву, авторів і короткий опис.

Для взаємодії з пошуком там необхідно створити `Tool`. В лекції ми використовували `serpapi`. Можна продовжити користуватись ним, або обрати інше АРІ для пошуку (вони в тому числі є безкоштовні). Перелік різних АРІ, доступних в langchain, і орієнтир по вартості запитів можна знайти в окремому документі [тут](https://hannapylieva.notion.site/API-12994835849480a69b2adf2b8441cbb3?pvs=4).

Лишаю також нижче приклад використання одного з безкоштовних пошукових АРІ - DuckDuckGo (не потребує створення токена!)  - можливо він вам сподобається :)


In [15]:
!pip install -q langchain_community duckduckgo_search

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

search.invoke("Obama's first name?")

"2 of 2. Barack Obama: timeline Key events in the life of Barack Obama. Barack Obama (born August 4, 1961, Honolulu, Hawaii, U.S.) is the 44th president of the United States (2009-17) and the first African American to hold the office. Before winning the presidency, Obama represented Illinois in the U.S. Senate (2005-08). Since the office was established in 1789, 45 men have served in 46 presidencies. The first president, George Washington, won a unanimous vote of the Electoral College. [4] Grover Cleveland served two non-consecutive terms and is therefore counted as the 22nd and 24th president of the United States, giving rise to the discrepancy between the ... Here is a list of the presidents and vice presidents of the United States along with their parties and dates in office. ... Chester A Arthur: Twenty-First President of the United States. 10 Interesting Facts About James Buchanan. Martin Van Buren - Eighth President of the United States. Quotes From Harry S. Truman. Table of Cont

In [16]:
!pip install -q google-search-results langchain-community langchain_experimental

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [20]:
!pip install mypy_extensions

Скористаємося Serpapi API

In [5]:
llm_agent = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-72B-Instruct",
    temperature = 0.1,
    max_new_tokens=500
)

chat_agent = ChatHuggingFace(llm=llm_agent)

In [13]:
tools = load_tools(["serpapi"], llm = llm_agent)

In [14]:
print(tools)

[Tool(name='Search', description='A search engine. Useful for when you need to answer questions about current events. Input should be a search query.', func=<bound method SerpAPIWrapper.run of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='e4eeda02ed55fc0b96d38bbe2333d702c1025deecb98314ea6bddacff042acbb', aiosession=None)>, coroutine=<bound method SerpAPIWrapper.arun of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='e4eeda02ed55fc0b96d38bbe2333d702c1025deecb98314ea6bddacff042acbb', aiosession=None)>)]


Завантажуємо промпт

In [6]:
from langsmith import Client

client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [7]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


In [15]:
agent = create_react_agent(chat_agent, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

In [57]:
request = "Find the 5 most recent scientific publications on the topic of artificial intelligence. For each one: title, authors, and a very short 1- to 2-sentence description."

In [58]:
response = agent_executor.invoke({"input": request})
print(response["output"])



> Entering new AgentExecutor chain...
I need to search for the most recent scientific publications on artificial intelligence.
Action: Search
Action Input: "5 most recent scientific publications on artificial intelligence"['12:00—Exploring AI. Aktion T4, a “Disposable Humanity” uncovers this largely forgotten history through a 25-year investigation by the Mitchell family. (315) ...', '668K followers · 949 following · 7458 posts · @most: “hello Gen Z lesbians (The Body) and millennial lesbians (The Hunting Wives)”', "MOST &mdash Missouri's 529 Education Plan is affordable, tax-advantaged, easy to join, and open to everyone.", 'Recently Published Science Notes. MOST Policy Fellows provide nonpartisan Science Notes to describe the research behind policy issues.', '27th APEC Policy Partnership on Science, Technology and Innovation (PPSTI) Meeting Successfully Convened · Action Plan for International Cooperation in Open ...', 'Created for humans, backed by science MOST effectively blends 

Бачимо, що результатом є заглушки. Потрібен пошук по архіву

In [11]:
!pip install -U -q arxiv

In [8]:
# Завантажуємо інструмент arXiv замість serpapi
tools = load_tools(["arxiv"])
print(tools)

agent = create_react_agent(chat_agent, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

# Додаємо орієнтир за часом, щоб агент не шукав старі статті
request = "Find the 5 most recent scientific publications on the topic of artificial intelligence published in 2026. For each one provide: title, authors, and a short 1- to 2-sentence description."

[ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=3, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000))]


In [9]:
response = agent_executor.invoke({"input": request})
print(response["output"])



> Entering new AgentExecutor chain...
I need to search for the most recent publications on the topic of artificial intelligence from 2026.
Action: arxiv
Action Input: "artificial intelligence" year:2026

AttributeError: 'Search' object has no attribute 'results'

Напишемо власний робочий інструмент

In [10]:
import arxiv
from langchain_core.tools import tool

# 1. Створюємо робочий інструмент замість load_tools(["arxiv"])
@tool
def search_arxiv(query: str) -> str:
    """Search arXiv for recent scientific papers. Input should be a simple topic string like 'artificial intelligence'."""
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=5,
        sort_by=arxiv.SortCriterion.SubmittedDate
    )
    results = []
    for paper in client.results(search):
        authors = ", ".join(a.name for a in paper.authors[:4])
        results.append(
            f"Title: {paper.title}\n"
            f"Published: {paper.published.strftime('%Y-%m-%d')}\n"
            f"Authors: {authors}\n"
            f"Abstract: {paper.summary[:400]}..."
        )
    return "\n\n---\n\n".join(results)

# 2. Передаємо наш інструмент в агента (використовуючи вашу змінну chat_agent)
tools = [search_arxiv]
print(tools)

agent = create_react_agent(chat_agent, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True, handle_parsing_errors=True)

request = "Find the 5 most recent scientific publications on the topic of artificial intelligence. For each one provide: title, authors, and a short 1- to 2-sentence description."


[StructuredTool(name='search_arxiv', description="Search arXiv for recent scientific papers. Input should be a simple topic string like 'artificial intelligence'.", args_schema=<class 'langchain_core.utils.pydantic.search_arxiv'>, func=<function search_arxiv at 0x7fb84cad4860>)]


In [11]:
response = agent_executor.invoke({"input": request})
print(response["output"])



> Entering new AgentExecutor chain...
I need to search arXiv for the most recent publications on artificial intelligence.
Action: search_arxiv
Action Input: 'artificial intelligence'Title: PlayCraft: Designing Engaging Games from Agent Play Experience
Published: 2026-11-19
Authors: Weihao Cheng
Abstract: Recent AI agents can generate playable games, but these games are not necessarily engaging. Mastery is a key source of engagement, yet designing for it requires extensive human play and tuning. We propose PlayCraft, an automated framework that uses a vision-based agent to play games and draws on its detailed play experience to revise them for mastery. Starting from an initial game, PlayCraft alter...

---

Title: One Prior, Two Dynamics: Repurposing Video Diffusion Transformers for Self-Supervised Latent Action Learning
Published: 2026-11-19
Authors: Yuxin Jiang, Wenqiang Sun, Jiaqi Li, Yuchao Gu
Abstract: Videos reveal how the world changes, but rarely the actions that cause those c

Отримали правдоподібний результат



### Завдання 4: Створення агента-помічника для вирішення бізнес-задач

Створіть агента, який допомагає вирішувати задачі бізнес-аналітики. Агент має допомогти користувачу створити прогноз по продажам на наступний рік враховуючи рівень інфляції і погодні умови. Агент має вміти використовувати Python і ходити в інтернет аби отримати актуальні дані.

**Кроки:**
1. Налаштуйте агента, який працюватиме з аналітичними даними, заданими текстом. Користувач пише

```
Ми експортуємо апельсини з Бразилії. В 2022 експортували 200т, в 2023 - 190т, в 2024 - 210т, в 2025 - 220т. Зроби оцінку скільки ми зможемо експортувати апельсинів в 2026 враховуючи погодні умови в Бразилії і попит на апельсини в світі виходячи з економічної ситуації.
```

2. Створіть запит до агента, що містить чітке завдання – видати результат бізнес аналізу або написати, що він не може цього зробити і запит користувача (просто може бути все одним повідомлленням).

3. Запустіть агента і проаналізуйте результати. Що можна покращити?


In [16]:
llm_business_agent = HuggingFaceEndpoint(
    repo_id = "Qwen/Qwen2.5-72B-Instruct",
    temperature = 0.1,
    max_new_tokens=500
)

chat_business_agent = ChatHuggingFace(llm=llm_business_agent)

In [19]:
tools = load_tools(["serpapi"], llm=chat_business_agent) # internet search tool

# Python tool
python_repl = PythonREPL()
python_tool = Tool(
    name="python_interpreter",
    func=python_repl.run,
    description="Use for any complex calculations, prime numbers, loops, or logic. Input must be valid Python code that uses print() to output the result."
)

tools.append(python_tool)

In [20]:
tools

[Tool(name='Search', description='A search engine. Useful for when you need to answer questions about current events. Input should be a search query.', func=<bound method SerpAPIWrapper.run of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='e4eeda02ed55fc0b96d38bbe2333d702c1025deecb98314ea6bddacff042acbb', aiosession=None)>, coroutine=<bound method SerpAPIWrapper.arun of SerpAPIWrapper(search_engine=<class 'serpapi.google_search.GoogleSearch'>, params={'engine': 'google', 'google_domain': 'google.com', 'gl': 'us', 'hl': 'en'}, serpapi_api_key='e4eeda02ed55fc0b96d38bbe2333d702c1025deecb98314ea6bddacff042acbb', aiosession=None)>),
 Tool(name='python_interpreter', description='Use for any complex calculations, prime numbers, loops, or logic. Input must be valid Python code that uses print() to output the result.', func=<bound method PythonREPL.run of PythonREPL(g

In [21]:
print(prompt.template)

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}


Доповнимо промпт правилами бізнес-аналізу

In [24]:
biz_rules = """
You are a business analyst. You receive a text-based request.

Algorithm:
1) Extract: metrics, units, geography, time horizon, historical figures.
2) If metrics or numbers are missing, ask up to 3 clarifying questions and stop.
3) Build a baseline in python_interpreter (trend/average/simple regression forecast).
4) Use Search to find relevant external factors for the region and industry (weather, demand, macroeconomics), with links.
5) Specify low/base/high scenarios, assumptions, risks, and how the factors influenced the baseline.

Rule for Rejection:
If it is impossible to substantiate the figures with data (no historical or geographical context, or the sources are contradictory),
state exactly:
“I cannot provide a substantiated forecast. Reasons: ... Data needed: ...”
"""

In [32]:
client = Client()
base_prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

In [35]:
tmpl = biz_rules.strip() + "\n\n" + base_prompt.template
business_prompt = PromptTemplate.from_template(tmpl)

In [36]:
print(business_prompt.template)

You are a business analyst. You receive a text-based request.

Algorithm:
1) Extract: metrics, units, geography, time horizon, historical figures.
2) If metrics or numbers are missing, ask up to 3 clarifying questions and stop.
3) Build a baseline in python_interpreter (trend/average/simple regression forecast).
4) Use Search to find relevant external factors for the region and industry (weather, demand, macroeconomics), with links.
5) Specify low/base/high scenarios, assumptions, risks, and how the factors influenced the baseline.

Rule for Rejection:
If it is impossible to substantiate the figures with data (no historical or geographical context, or the sources are contradictory),
state exactly:
“I cannot provide a substantiated forecast. Reasons: ... Data needed: ...”

Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to 

In [39]:
business_agent = create_react_agent(chat_business_agent, tools, business_prompt)
business_executor = AgentExecutor(agent=business_agent, tools=tools, verbose=True)

In [40]:
business_question = "We export oranges from Brazil. In 2022, we exported 200 metric tons; in 2023, 190 metric tons; in 2024, 210 metric tons; and in 2025, 220 metric tons. Estimate how many metric tons of oranges we will be able to export in 2026, taking into account weather conditions in Brazil and global demand for oranges based on the economic situation in Brazil and Ukraine."

In [41]:
business_response = business_executor.invoke({"input": business_question})
print(business_response["output"])



> Entering new AgentExecutor chain...


I need to extract the metrics, units, geography, time horizon, and historical figures. Then, I will check if any additional information is needed.

Metrics: Export volume
Units: Metric tons
Geography: Brazil
Time Horizon: 2026
Historical Figures: 2022 - 200 metric tons, 2023 - 190 metric tons, 2024 - 210 metric tons, 2025 - 220 metric tons

I will now build a baseline forecast using the historical data.
Action: python_interpreter
Action Input:
```python
import numpy as np
from sklearn.linear_model import LinearRegression

# Historical data
years = np.array([2022, 2023, 2024, 2025]).reshape(-1, 1)
exports = np.array([200, 190, 210, 220])

# Linear regression model
model = LinearRegression()
model.fit(years, exports)

# Predict for 2026
predicted_2026 = model.predict([[2026]])
print(predicted_2026[0])
```225.0
The baseline forecast for 2026 is 225 metric tons of oranges. Next, I need to consider external factors such as weather conditions in Brazil and global demand for oranges based on 

Загалом "роздуми" агента виглядають логічно, проте з логів не повністю зрозуміло, як саме агент визначав результати для різних сценаріїв. Теоретично, це можна додати в промпт як додаткову інструкцію.
Також потрібно врахувати, що температура та кількість токенів на виході для моделі дуже обмежені.

Інший нюанс - саме по собі питання має досить мало контексту для отримання якісної відповіді:
- особливості компанії та експорту, які бізнес-аналітик може знати, проте агент не знає;
- малий обсяг даних на вході -> для моделі лінійної регресії був виконаний лише фіт без підбору гіперпараметрів, даних для оцінки якості моделі замало -> результат спрогнозовано ненадійно з точки зору машинного навчання.
- Для кращого результату було б добре під'єднати агента до бази даних компанії, додати тулзи для роботи з базою даних.
- Також в ідеалі дати агенту більше контексту та специфіки самої роботи компанії - під'єднати до векторної бази даних, сформованої з необхідної документації компанії